# DATA 266 - Homework 10: Improving RAG with Reranking, Chunking, and Conversational Memory

**Goal.** Take the RAG pipeline from the Demo 10 notebooks and check three ways of making the *retrieval* part better:

1. **Chunking.** Split the papers in two different ways and see which one gives better chunks to retrieve.
2. **Reranking.** Take the top 20 chunks from ChromaDB and reorder them with a cross-encoder.
3. **Conversational memory.** Ask a follow-up question that only makes sense with the previous turn ("that method", "this approach") and see whether memory fixes the retrieval.

Everything is measured with **Precision@K, Recall@K and MRR** using relevance rules I wrote by hand after reading the papers.

**Data.** `ccdv/arxiv-summarization`, config `document` (the full paper text, not the pre-chunked `section` config), split `train`, **only the first 100 documents**.

**What I took from the demos:**

- **`Demo_10_RAG_Chatbot_Local.ipynb`** is the main base. From it I reused the local stack (ChromaDB through `langchain_chroma`, `HuggingFaceEmbeddings` with `mixedbread-ai/mxbai-embed-large-v1`, `ChatOllama`), the batched indexing loop, `similarity_search`, the `augment_prompt` idea ("Using the contexts below, answer the query"), the message-list chat history, and the LangChain imports for memory (`InMemoryChatMessageHistory`, `RunnableWithMessageHistory`, `ChatPromptTemplate`, `MessagesPlaceholder`) and for chunking (`RecursiveCharacterTextSplitter`). The demo imports these but never actually uses them, so this homework is where they get used.
- **`Demo_10_RAG_Chatbot.ipynb`** is the same pipeline with Pinecone and Gemini/OpenAI. I took the idea of storing metadata next to each vector (title/arxiv id there, paper id and chunk index here), but I didn't use Pinecone or the paid APIs, so the notebook needs no API keys.
- **`Demo_10_cagdemo.ipynb`** shows cache-augmented generation: the whole document is put into Mistral-7B's KV cache once, and `clean_up()` crops the cache back after every question. I didn't use it for the experiments. A 7B model in fp32 doesn't fit in my M1's 8 GB, and the homework is about retrieval. CAG also skips retrieval completely, so there would be nothing to rerank. The idea of keeping state between questions is related to the memory part, though.

**What I changed from the demos, and why:**

- **The demo indexes title + abstract as one "chunk".** Here I index the paper body (`article`) and split it myself, because chunking is part of the assignment. I keep the abstract only for inspection.
- **Persistent ChromaDB instead of a Chroma server.** The demo connects to `host="localhost"`, which needs a separate server running. I use `persist_directory`, so the two collections are saved to a folder and a re-run reuses the vectors instead of embedding everything again.
- **The demo hard-codes `cuda:1`.** I pick CUDA, then Apple MPS, then CPU automatically.
- **LLM.** The Local demo uses `phi3:mini` in Ollama. I already had `llama3.2:3b` pulled in Ollama and my disk was almost full, so I used that instead. The LLM is only used for the memory part (rewriting the follow-up question and writing short answers). All the metrics are about retrieval.

## 1. Personal Parameters

Same formulas as in my earlier assignments. The assert checks that the values match the ones I've used all semester.

- **SEED** seeds Python, NumPy and PyTorch, and it's also passed to Ollama so the LLM rewrites and answers are repeatable.
- **SLICE** picks the paper I use to show the chunk examples: paper `SLICE % 100` = 86. The dataset rows themselves are fixed by the assignment (the first 100), so SLICE can't be used to pick the data here.
- **HP_ID, CLS_A, CLS_B** are only reported. This homework has no classes or hyperparameter grid to pick from.

In [1]:
SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

assert (SEED, SLICE, HP_ID, CLS_A, CLS_B) == (9486, 486, 0, 6, 0)

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")

SID4  = 9486
SEED  = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0


## 2. Package Setup

The Local demo starts with `!pip install -qU chromadb langchain-community langchain-ollama langchain-chroma langchain-huggingface`. I only install what's missing and I don't use `-U`, because upgrading everything can break an existing torch/transformers setup. `langchain-community` isn't needed (it's being sunset and I don't load PDFs), and `sentence-transformers` is added for the cross-encoder reranker.

The LLM part needs [Ollama](https://ollama.com/) running locally with the model pulled (`ollama pull llama3.2:3b`). If Ollama isn't running, the notebook still runs everything else and says clearly which LLM steps were skipped.

In [2]:
import importlib.util
import subprocess
import sys

needed = {
    "datasets": "datasets",
    "chromadb": "chromadb",
    "langchain_core": "langchain-core",
    "langchain_chroma": "langchain-chroma",
    "langchain_huggingface": "langchain-huggingface",
    "langchain_text_splitters": "langchain-text-splitters",
    "langchain_ollama": "langchain-ollama",
    "sentence_transformers": "sentence-transformers",
}
missing = [pip_name for module, pip_name in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("all packages are already installed")

all packages are already installed


## 3. Imports and Reproducibility

In [3]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # avoids fork warnings from the HF tokenizers
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")     # turns off ChromaDB telemetry

import json
import random
import re
import textwrap
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm  # for progress bar

from datasets import load_dataset

from langchain_core.chat_history import (
    BaseChatMessageHistory,      # interface for pluggable chat history backends
    InMemoryChatMessageHistory,  # in-memory chat history for one session
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from sentence_transformers import CrossEncoder

import chromadb
import datasets
import langchain_core
import sentence_transformers
import transformers

transformers.logging.set_verbosity_error()   # hides the "sequence longer than 512" tokenizer warning, I check lengths myself
warnings.filterwarnings("ignore", category=FutureWarning)
try:   # LangChain 1.7 marks the demo's memory classes as deprecated (see Section 13), they still work fine
    from langchain_core._api import LangChainDeprecationWarning
    warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
except ImportError:
    pass
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

W1110 10:59:24.483000 56056 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### Experiment settings

All the settings are in one place, so it's easy to see what each experiment used.

- **Chunking.** Strategy 1 uses 200-word windows with a 40-word overlap. Strategy 2 uses recursive, sentence-aware chunks of at most 1,200 characters with up to 200 characters of overlap. I picked the sizes so both strategies give chunks of roughly the same length (about 200 words). That way the comparison is mostly about *where* the chunks are cut and how the overlap works, not about one strategy just having bigger chunks.
- **Retrieval.** ChromaDB returns `N_CANDIDATES = 20` chunks per query. Without reranking I take the top K straight from that list. With reranking, the cross-encoder reorders the same 20 and I take its top K, so both use exactly the same candidate pool.
- **Metrics.** K = 3 and K = 5 for Precision@K and Recall@K. MRR looks at the first 10 results (MRR@10).
- **Smoke test vs. full run.** `SMOKE_TEST` comes from the environment variable `HW10_SMOKE_TEST` and is off by default, so Run All does the real experiment. With `HW10_SMOKE_TEST=1` every paper is cut to its first 300 words and saved in a separate ChromaDB folder. That's only to check that every cell runs in a couple of minutes, and the smoke-test metrics mean nothing.

In [4]:
SMOKE_TEST = os.environ.get("HW10_SMOKE_TEST", "0") == "1"

# data (fixed by the assignment)
DATASET_NAME = "ccdv/arxiv-summarization"
DATASET_CONFIG = "document"
DATASET_SPLIT = "train"
N_DOCS = 100
EXAMPLE_DOC = SLICE % N_DOCS          # paper used for the chunk examples (86)

# chunking strategy 1: fixed-size word windows
S1_WORDS = 200
S1_OVERLAP = 40
# chunking strategy 2: recursive, sentence-aware character chunks
S2_CHARS = 1200
S2_OVERLAP = 200
S2_SEPARATORS = [". ", "; ", ", ", " ", ""]

# embedding model (same as Demo_10_RAG_Chatbot_Local) and a small fallback
EMBED_MODEL = os.environ.get("HW10_EMBED_MODEL", "mixedbread-ai/mxbai-embed-large-v1")
EMBED_FALLBACK = "BAAI/bge-small-en-v1.5"
QUERY_PROMPT = "Represent this sentence for searching relevant passages: "  # mxbai's query instruction
EMBED_BATCH = 16       # chunks per forward pass of the embedding model
INDEX_BATCH = 256      # chunks per add_texts call to ChromaDB

# reranker
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
RERANK_FALLBACK = "cross-encoder/ms-marco-TinyBERT-L2-v2"

# retrieval and evaluation
N_CANDIDATES = 20
K_VALUES = (3, 5)
MRR_DEPTH = 10
SHOW_K = 5

# conversational memory
LLM_MODEL = os.environ.get("HW10_LLM", "llama3.2:3b")
CHAT_STRATEGY = "s2"   # collection used for the chatbot answers (all combinations are evaluated in Section 15)
N_CONTEXT = 3          # chunks given to the LLM, like the demo's k=3

CHROMA_DIR = "chroma_db"
SMOKE_MAX_WORDS = None
if SMOKE_TEST:
    SMOKE_MAX_WORDS = 300
    CHROMA_DIR = "chroma_db_smoke"

CONFIG = dict(
    smoke_test=SMOKE_TEST, dataset=DATASET_NAME, config=DATASET_CONFIG, split=DATASET_SPLIT, n_docs=N_DOCS,
    s1_words=S1_WORDS, s1_overlap=S1_OVERLAP, s2_chars=S2_CHARS, s2_overlap=S2_OVERLAP, s2_separators=S2_SEPARATORS,
    embed_model=EMBED_MODEL, rerank_model=RERANK_MODEL, n_candidates=N_CANDIDATES, k_values=list(K_VALUES),
    mrr_depth=MRR_DEPTH, llm=LLM_MODEL, chat_strategy=CHAT_STRATEGY, seed=SEED,
)
print(json.dumps(CONFIG, indent=1))

{
 "smoke_test": false,
 "dataset": "ccdv/arxiv-summarization",
 "config": "document",
 "split": "train",
 "n_docs": 100,
 "s1_words": 200,
 "s1_overlap": 40,
 "s2_chars": 1200,
 "s2_overlap": 200,
 "s2_separators": [
  ". ",
  "; ",
  ", ",
  " ",
  ""
 ],
 "embed_model": "mixedbread-ai/mxbai-embed-large-v1",
 "rerank_model": "cross-encoder/ms-marco-MiniLM-L6-v2",
 "n_candidates": 20,
 "k_values": [
  3,
  5
 ],
 "mrr_depth": 10,
 "llm": "llama3.2:3b",
 "chat_strategy": "s2",
 "seed": 9486
}


### Device, seeds and logging

- The device is picked automatically: **CUDA** first, then **Apple MPS** (my M1), then **CPU**. `HW10_DEVICE=cpu` (or `mps`, `cuda`) forces one.
- `set_seed` seeds random, NumPy and PyTorch. Nothing is trained here, and embedding and reranking are deterministic forward passes, so the seed mostly matters for the LLM. Ollama gets `seed=SEED` and `temperature=0`.
- `log()` prints each line with a UTC timestamp and also writes it to `results/run_log_raw.txt`.

In [5]:
for folder in ["results"]:
    os.makedirs(folder, exist_ok=True)
LOG_PATH = "results/run_log_raw_smoke.txt" if SMOKE_TEST else "results/run_log_raw.txt"
open(LOG_PATH, "w").close()


def log(*args):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    line = f"[{stamp}] " + " ".join(str(a) for a in args)
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


set_seed(SEED)
device = os.environ.get("HW10_DEVICE") or get_device()
if device == "cuda":
    device_name = torch.cuda.get_device_name(0)
elif device == "mps":
    device_name = "Apple GPU (MPS)"
else:
    device_name = "CPU only"

RUN_START = time.time()
log("device:", device, "|", device_name, "| smoke test:", SMOKE_TEST)
log("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
    "| sentence-transformers", sentence_transformers.__version__, "| chromadb", chromadb.__version__,
    "| langchain-core", langchain_core.__version__, "| datasets", datasets.__version__)

[2026-11-10T18:59:26Z] device: mps | Apple GPU (MPS) | smoke test: False
[2026-11-10T18:59:26Z] python 3.13.9 | torch 2.14.0 | transformers 5.21.0 | sentence-transformers 6.2.0 | chromadb 1.5.9 | langchain-core 1.7.7 | datasets 5.1.0


## 4. Dataset Loading and Inspection

The demo uses `load_dataset(...)` and then works with the whole split. The full `ccdv/arxiv-summarization` train split has 203,037 papers in 15 parquet shards (several GB), and I only need the first 100, so I load it with `streaming=True` and `take(N_DOCS)`. This reads the rows in their original order from the first shard and stops after 100. Before writing the notebook I checked rows 0-4 and 95-99 against the Hugging Face dataset viewer API, and they were the same papers.

Each row has two fields: `article` (the full paper body) and `abstract`.

In [6]:
t0 = time.time()
stream = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT, streaming=True)
raw_docs = list(stream.take(N_DOCS))
assert len(raw_docs) == N_DOCS, f"expected {N_DOCS} documents, got {len(raw_docs)}"
log(f"loaded {len(raw_docs)} documents from {DATASET_NAME} [{DATASET_CONFIG}] split={DATASET_SPLIT} in {time.time() - t0:.1f}s")
log("fields:", list(raw_docs[0].keys()))

overview = pd.DataFrame({
    "doc_id": range(N_DOCS),
    "article_words": [len(d["article"].split()) for d in raw_docs],
    "abstract_words": [len(d["abstract"].split()) for d in raw_docs],
    "abstract_start": [d["abstract"].strip()[:90] for d in raw_docs],
})
print(overview.head(10).to_string(index=False))
print()
print(overview[["article_words", "abstract_words"]].describe().round(1).to_string())
print("\ntotal article words:", overview.article_words.sum())

[2026-11-10T18:59:29Z] loaded 100 documents from ccdv/arxiv-summarization [document] split=train in 3.7s
[2026-11-10T18:59:29Z] fields: ['article', 'abstract']
 doc_id  article_words  abstract_words                                                                              abstract_start
      0           4467             145 additive models play an important role in semiparametric statistics . \n this paper gives l
      1           3223              61  we have studied the leptonic decay @xmath0 , via the decay channel @xmath1 , using a sampl
      2           5292             260  in 84 , 258 ( 2000 ) , mateos conjectured that current reversal in a classical determinist
      3           3714              81  the effect of a random phase diffuser on fluctuations of laser light ( scintillations ) is
      4           9235             151  with a special intention of clarifying the underlying spin contents of the nucleon , we in
      5           6725              66  we improve the

In [7]:
print("Raw text of paper", EXAMPLE_DOC, "(first 700 characters):\n")
print(textwrap.fill(raw_docs[EXAMPLE_DOC]["article"][:700], 110))
print("\nnewlines inside the 100 articles:", sum(d["article"].count("\n") for d in raw_docs))

Raw text of paper 86 (first 700 characters):

we first classify the multiplicity - free permutation characters given by the actions of symmetric groups on
their conjugacy classes . for this we shall need the following lemma , ultimately due to frobenius , which
implies that multiplicity - free permutation characters only come from permutation actions with relatively
high degrees of homogeneity . [ lemma : tworow ] let @xmath95 be a permutation group acting on @xmath96 . let
@xmath5 be the permutation character of the action of @xmath1 on the cosets of  @xmath37 . let @xmath97 be the
number of orbits of @xmath37 on @xmath98-subsets of @xmath96 . if  @xmath99 then @xmath100    we shall also
need the forms of young s rule and pieri s rule

newlines inside the 100 articles: 0


**What I noticed:** the papers have very different lengths, from about 600 words up to about 33,000. The text has already been processed by the dataset authors: it's all lowercase, there's a space before every punctuation mark (`word .`), equations are replaced by `@xmath0`, `@xmath1`, ..., and citations by `@xcite`. The articles also have **no newlines at all**, so there are no paragraph breaks to split on. This affects the sentence-aware chunker below, because the only boundaries it can use are sentence endings.

## 5. Preprocessing

I kept the cleaning light, because I don't want to change the content:

1. `@xmath12` becomes `[math]` and `@xcite` becomes `[cite]`. The numbers after `@xmath` are just placeholder ids. They mean nothing to the embedding model and they take up tokens.
2. Spaced hyphens are joined back (`non - invasive` becomes `non-invasive`), and the extra space before punctuation and after brackets is removed (`word .` becomes `word.`). This makes the text closer to normal English, which the embedding model and the reranker were trained on.
3. All whitespace is collapsed to single spaces.

I don't lowercase anything, because the text is already lowercase. I don't remove stop words either, since sentence embeddings need full sentences. All 100 papers are kept, including the very short and very long ones.

In [8]:
def clean_text(text):
    text = re.sub(r"@xmath\d+", "[math]", text)
    text = text.replace("@xcite", "[cite]")
    text = re.sub(r"(\w) - (\w)", r"\1-\2", text)           # non - invasive -> non-invasive
    text = re.sub(r" ([.,;:?!)\]])", r"\1", text)            # word . -> word.
    text = re.sub(r"([(\[]) ", r"\1", text)                  # ( word -> (word
    return re.sub(r"\s+", " ", text).strip()


docs = []
for i, d in enumerate(raw_docs):
    text = clean_text(d["article"])
    if SMOKE_MAX_WORDS:
        text = " ".join(text.split()[:SMOKE_MAX_WORDS])
    docs.append({"doc_id": i, "text": text, "abstract": clean_text(d["abstract"])})

print("before:", raw_docs[33]["article"][:330])
print("after: ", docs[33]["text"][:300])
n_words = [len(d["text"].split()) for d in docs]
log(f"cleaned {len(docs)} papers | words per paper: min {min(n_words)}, mean {np.mean(n_words):.0f}, "
    f"max {max(n_words)}, total {sum(n_words):,} | characters total {sum(len(d['text']) for d in docs):,}")

before: high q cavities such as whispering gallery mode ( wgm ) cavities have recently demonstrated quality factors ( @xmath0 ) as high as @xmath3 and have shown the potential to reach even higher q values @xcite . however , there are difficulties in measurement of the linewidth and q of such high q cavities . while in theory , the q fa
after:  high q cavities such as whispering gallery mode (wgm) cavities have recently demonstrated quality factors ([math]) as high as [math] and have shown the potential to reach even higher q values [cite]. however, there are difficulties in measurement of the linewidth and q of such high q cavities. while
[2026-11-10T18:59:30Z] cleaned 100 papers | words per paper: min 528, mean 5419, max 27600, total 541,857 | characters total 3,407,419


## 6. Chunking Strategy 1: Fixed-Size Word Windows with Overlap

The simplest option. Each paper is turned into a list of words, and a 200-word window slides over it with a step of 160 words, so neighbouring chunks share 40 words (20% overlap). The windows completely ignore sentences. A chunk can start or end in the middle of a sentence, or even in the middle of an equation placeholder.

Why these numbers:
- **200 words** come to about 320 tokens with the mxbai tokenizer (I check this in Section 9). That fits under the model's 512-token limit for most chunks, and it's about one paragraph of a paper, which is a sensible amount of context to give an LLM.
- **40 words of overlap** means a sentence that gets cut at the end of one chunk still appears whole at the start of the next one most of the time.
- The last window of a paper may be shorter. I don't pad it or merge it, so short leftovers show up in the statistics.

In [9]:
def chunk_fixed_words(text, size=S1_WORDS, overlap=S1_OVERLAP):
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks


def make_records(strategy, doc, pieces):
    return [{
        "chunk_id": f"{strategy}-d{doc['doc_id']:03d}-c{j:04d}",
        "strategy": strategy,
        "doc_id": doc["doc_id"],
        "chunk_index": j,
        "text": piece,
        "n_words": len(piece.split()),
        "n_chars": len(piece),
    } for j, piece in enumerate(pieces)]


chunks_s1 = [r for d in docs for r in make_records("s1", d, chunk_fixed_words(d["text"]))]
log(f"strategy 1 (fixed {S1_WORDS} words, overlap {S1_OVERLAP}): {len(chunks_s1):,} chunks")

[2026-11-10T18:59:30Z] strategy 1 (fixed 200 words, overlap 40): 3,411 chunks


## 7. Chunking Strategy 2: Recursive, Sentence-Aware Chunks

This one uses `RecursiveCharacterTextSplitter`, which the demos import but never use. It tries the separators in order: first it splits the text at sentence endings (`". "`). If a piece is still longer than 1,200 characters, it splits that piece at `"; "`, then `", "`, then spaces. After that it **merges whole sentences back together** until adding the next sentence would go over 1,200 characters.

So compared to strategy 1:
- **The boundary** is a sentence ending (almost always), not an arbitrary word count.
- **The size** is measured in characters, not words, so chunks with lots of `[math]` placeholders hold fewer real words.
- **The overlap** works differently too. The splitter carries over whole sentences from the end of the previous chunk, as long as they fit into 200 characters. If the last sentence is longer than 200 characters, there's no overlap at all. In strategy 1 the overlap is always exactly 40 words.

`keep_separator="end"` keeps the period at the end of each sentence instead of moving it to the start of the next chunk.

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=S2_CHARS,
    chunk_overlap=S2_OVERLAP,
    separators=S2_SEPARATORS,
    keep_separator="end",
    length_function=len,
)

chunks_s2 = [r for d in docs for r in make_records("s2", d, [p.strip() for p in splitter.split_text(d["text"])])]
log(f"strategy 2 (recursive {S2_CHARS} chars, overlap up to {S2_OVERLAP}): {len(chunks_s2):,} chunks")

[2026-11-10T18:59:30Z] strategy 2 (recursive 1200 chars, overlap up to 200): 3,500 chunks


## 8. Chunk Statistics and Comparison

For the overlap of strategy 2 I measure what actually happened: for every pair of neighbouring chunks from the same paper, I find the longest piece of text that ends chunk *i* and also starts chunk *i+1*.

In [11]:
def actual_overlap_chars(a, b, max_len=600):
    for n in range(min(len(a), len(b), max_len), 0, -1):
        if a.endswith(b[:n]):
            return n
    return 0


def overlap_stats(chunks):
    out = []
    for prev, nxt in zip(chunks, chunks[1:]):
        if prev["doc_id"] == nxt["doc_id"]:
            out.append(actual_overlap_chars(prev["text"], nxt["text"]))
    return np.array(out)


def ends_at_sentence(t):
    return t.rstrip().endswith((".", "?", "!"))


rows = []
for name, chunks, setting in [("S1 fixed words", chunks_s1, f"{S1_WORDS} words, {S1_OVERLAP}-word overlap"),
                              ("S2 recursive sentences", chunks_s2, f"<= {S2_CHARS} chars, <= {S2_OVERLAP}-char overlap")]:
    w = np.array([c["n_words"] for c in chunks])
    ch = np.array([c["n_chars"] for c in chunks])
    ov = overlap_stats(chunks)
    per_doc = pd.Series([c["doc_id"] for c in chunks]).value_counts()
    rows.append({
        "strategy": name, "settings": setting, "chunks": len(chunks),
        "words min": w.min(), "words mean": round(w.mean(), 1), "words max": w.max(),
        "chars min": ch.min(), "chars mean": round(ch.mean(), 1), "chars max": ch.max(),
        "overlap chars mean": round(ov.mean(), 1), "pairs with no overlap %": round(100 * (ov == 0).mean(), 1),
        "ends at sentence %": round(100 * np.mean([ends_at_sentence(c["text"]) for c in chunks]), 1),
        "chunks/paper min-max": f"{per_doc.min()}-{per_doc.max()}",
    })
chunk_stats = pd.DataFrame(rows).set_index("strategy")
print(chunk_stats.T.to_string())

strategy                             S1 fixed words              S2 recursive sentences
settings                 200 words, 40-word overlap  <= 1200 chars, <= 200-char overlap
chunks                                         3411                                3500
words min                                        41                                   1
words mean                                    197.7                               172.2
words max                                       200                                 419
chars min                                       191                                   8
chars mean                                   1241.6                              1079.5
chars max                                      3368                                1199
overlap chars mean                            249.7                               109.3
pairs with no overlap %                         0.1                                21.2
ends at sentence %              

In [12]:
def show_boundary(c, width=170):
    t = c["text"]
    return f"   START: {t[:width]!r}\n   END:   {t[-width:]!r}"


print(f"Paper {EXAMPLE_DOC} (EXAMPLE_DOC = SLICE % 100). Abstract: {docs[EXAMPLE_DOC]['abstract'][:200]}...\n")
for name, chunks in [("Strategy 1", chunks_s1), ("Strategy 2", chunks_s2)]:
    mine = [c for c in chunks if c["doc_id"] == EXAMPLE_DOC]
    print(f"===== {name}: {len(mine)} chunks for this paper; first two shown =====")
    for c in mine[:2]:
        print(f"{c['chunk_id']}  ({c['n_words']} words, {c['n_chars']} chars)")
        print(show_boundary(c))
    print(f"   overlap between chunk 0 and chunk 1: {actual_overlap_chars(mine[0]['text'], mine[1]['text'])} characters\n")

Paper 86 (EXAMPLE_DOC = SLICE % 100). Abstract: building on work of saxl, we classify the multiplicity-free permutation characters of all symmetric groups of degree [math] or more. a corollary is a complete list of the irreducible characters of sym...

===== Strategy 1: 35 chunks for this paper; first two shown =====
s1-d086-c0000  (200 words, 1158 chars)
   START: 'we first classify the multiplicity-free permutation characters given by the actions of symmetric groups on their conjugacy classes. for this we shall need the following l'
   END:   'us later see theorem [thm: jamesyoung] in 3 below.) _ proof. _ that [math] is multiplicity-free in cases (i) and (ii) follows from young s rule, while case (iii) is given'
s1-d086-c0001  (200 words, 1174 chars)
   START: 'by james in this reference will be useful to us later see theorem [thm: jamesyoung] in 3 below.) _ proof. _ that [math] is multiplicity-free in cases (i) and (ii) follows'
   END:   'umber of orbits of [math] on unordered pair

**What the statistics show:** the two strategies give a similar number of chunks (3,411 vs. 3,500), and S2's chunks are a bit smaller on average (172 vs. 198 words), because it stops before a sentence that would go over 1,200 characters. The real difference is the boundaries. Only **7.0%** of S1 chunks end at the end of a sentence, compared with **97.6%** for S2. S1 always has its 40-word overlap (about 250 characters, and only 0.1% of pairs have none, at the very end of a paper). S2's overlap is shorter (109 characters on average), and **21.2%** of neighbouring pairs have no overlap at all, because the last sentence was longer than 200 characters. S2 also produces some very small chunks (minimum 1 word, from a reference list where the splitter had to fall back to commas), and its maximum is 419 words for chunks made mostly of short `[math]` tokens. S1's word count is fixed, but its character length goes up to 3,368 when a window covers dense math. The example from paper 86 shows the difference: the S1 chunks start and end mid-sentence, while the S2 chunks end with a full stop.

## 9. Embedding Model Setup

I use the same embedding model as both RAG demos, **`mixedbread-ai/mxbai-embed-large-v1`**: a 335M-parameter BERT-large style model that outputs 1024-dimensional vectors and has a 512-token limit. Two settings are different from the demo:

- `normalize_embeddings=True`, so every vector has length 1 and the cosine distance in ChromaDB is simply 1 − dot product.
- **Query instruction.** mxbai is trained to get the prefix `"Represent this sentence for searching relevant passages: "` on *queries* (not on documents). The demo leaves it out. `query_encode_kwargs` adds it only when a query is embedded.

If the model can't be downloaded, the notebook falls back to `BAAI/bge-small-en-v1.5` and logs it (that didn't happen in my run, see the output). I ran this on my M1 with MPS. The model takes about 0.3 s per chunk there (3.3-3.4 chunks/s), so indexing both strategies is the slowest part of the notebook (about 35 minutes in my run). On a CUDA GPU it would take a couple of minutes.

In [13]:
def load_embeddings(name):
    return HuggingFaceEmbeddings(
        model_name=name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True, "batch_size": EMBED_BATCH},
        query_encode_kwargs={"normalize_embeddings": True, "prompt": QUERY_PROMPT},
    )


t0 = time.time()
try:
    embeddings = load_embeddings(EMBED_MODEL)
    EMBED_USED = EMBED_MODEL
except Exception as e:
    log(f"could not load {EMBED_MODEL} ({type(e).__name__}: {e}); falling back to {EMBED_FALLBACK}")
    embeddings = load_embeddings(EMBED_FALLBACK)
    EMBED_USED = EMBED_FALLBACK
st_model = embeddings._client            # the underlying SentenceTransformer
MAX_TOKENS = st_model.max_seq_length
log(f"embedding model {EMBED_USED} loaded in {time.time() - t0:.1f}s on {device} | max tokens {MAX_TOKENS} | "
    f"parameters {sum(p.numel() for p in st_model.parameters()):,}")

# same check as the demo
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]
res = embeddings.embed_documents(texts)
dimensions = len(res[0])
print("demo check:", (len(res), dimensions), "| vector norm:", round(float(np.linalg.norm(res[0])), 4))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[2026-11-10T18:59:35Z] embedding model mixedbread-ai/mxbai-embed-large-v1 loaded in 4.6s on mps | max tokens 512 | parameters 335,141,888


demo check: (2, 1024) | vector norm: 1.0004


Next I check how many tokens the chunks really have. Anything over 512 tokens gets cut off by the model, so the end of that chunk doesn't affect its embedding. The `[math]` placeholders cost about 3 tokens each, which is why some 200-word chunks go over the limit.

In [14]:
tok = st_model.tokenizer
token_rows = []
TOKEN_COUNTS = {}
for name, chunks in [("s1", chunks_s1), ("s2", chunks_s2)]:
    counts = np.array([len(ids) for ids in tok([c["text"] for c in chunks], add_special_tokens=True)["input_ids"]])
    TOKEN_COUNTS[name] = counts
    token_rows.append({"strategy": name, "tokens mean": round(counts.mean(), 1), "tokens median": int(np.median(counts)),
                       "tokens max": counts.max(), f"chunks over {MAX_TOKENS}": int((counts > MAX_TOKENS).sum()),
                       "% truncated": round(100 * (counts > MAX_TOKENS).mean(), 2)})
print(pd.DataFrame(token_rows).to_string(index=False))

strategy  tokens mean  tokens median  tokens max  chunks over 512  % truncated
      s1        337.2            309        1717              179         5.25
      s2        293.1            276         851              137         3.91


## 10. ChromaDB Indexing

Like the Local demo, I use `langchain_chroma.Chroma`. The only difference is `persist_directory` instead of a server. **Each strategy gets its own collection** (`hw10-s1-fixed-words` and `hw10-s2-recursive`), so their chunks can never mix. Chunk ids also start with the strategy name (`s1-d027-c0012`), and every chunk has metadata: `strategy`, `doc_id`, `chunk_index`, word and character counts, and the source dataset/config/split.

- The collections use **cosine distance** (`hnsw.space = cosine`). ChromaDB returns the distance, and similarity = 1 − distance.
- The demo loops over the data in batches and calls `add_documents`. I do the same with `add_texts` and batches of 256 chunks. Inside each batch the embedding model runs 16 chunks at a time.
- **Re-runs.** Each collection stores a small fingerprint in its metadata: the embedding model, chunk settings and number of chunks. If the fingerprint matches, only the missing chunk ids are added (none on a normal re-run, so nothing gets embedded twice). If anything changed, the collection is deleted and built again. This also means an interrupted run can continue where it stopped.

In [15]:
COLLECTIONS = {"s1": "hw10-s1-fixed-words", "s2": "hw10-s2-recursive"}
CHUNKS = {"s1": chunks_s1, "s2": chunks_s2}
CHUNK_PARAMS = {"s1": f"fixed words={S1_WORDS} overlap={S1_OVERLAP}",
                "s2": f"recursive chars={S2_CHARS} overlap={S2_OVERLAP} sep={S2_SEPARATORS}"}
if SMOKE_TEST:
    COLLECTIONS = {k: v + "-smoke" for k, v in COLLECTIONS.items()}


def open_store(strategy):
    chunks = CHUNKS[strategy]
    fingerprint = {"embed_model": EMBED_USED, "strategy": strategy, "chunk_params": CHUNK_PARAMS[strategy],
                   "n_chunks": len(chunks), "n_docs": N_DOCS, "smoke_test": SMOKE_TEST}

    def make():
        return Chroma(
            collection_name=COLLECTIONS[strategy],
            embedding_function=embeddings,
            persist_directory=CHROMA_DIR,
            collection_metadata=fingerprint,
            collection_configuration={"hnsw": {"space": "cosine", "ef_construction": 200, "ef_search": 200}},
        )

    store = make()
    old = store._collection.metadata or {}
    if any(old.get(k) != v for k, v in fingerprint.items()):
        log(f"[{strategy}] collection settings changed (or new collection) -> rebuilding {COLLECTIONS[strategy]}")
        store.delete_collection()
        store = make()
    return store


def index_chunks(strategy):
    store = open_store(strategy)
    chunks = CHUNKS[strategy]
    have = set(store._collection.get(include=[])["ids"])
    todo = [c for c in chunks if c["chunk_id"] not in have]
    log(f"[{strategy}] collection {COLLECTIONS[strategy]}: {len(have):,} chunks already stored, {len(todo):,} to embed")
    t0 = time.time()
    for i in tqdm(range(0, len(todo), INDEX_BATCH), desc=f"indexing {strategy}"):
        batch = todo[i:i + INDEX_BATCH]
        store.add_texts(
            texts=[c["text"] for c in batch],
            metadatas=[{"strategy": c["strategy"], "doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                        "n_words": c["n_words"], "n_chars": c["n_chars"],
                        "source": f"{DATASET_NAME}/{DATASET_CONFIG}/{DATASET_SPLIT}"} for c in batch],
            ids=[c["chunk_id"] for c in batch],
        )
    took = time.time() - t0
    count = store._collection.count()
    assert count == len(chunks), f"{strategy}: collection has {count} chunks, expected {len(chunks)}"
    rate = f"{len(todo) / took:.1f} chunks/s" if todo else "nothing new to embed"
    log(f"[{strategy}] done: {count:,} chunks in the collection | {took:.1f}s for {len(todo):,} new chunks ({rate})")
    return store, took, len(todo)


INDEX_TIME = {}
STORES = {}
for s in ["s1", "s2"]:
    STORES[s], took, n_new = index_chunks(s)
    INDEX_TIME[s] = {"seconds": round(took, 1), "embedded_this_run": n_new}

[2026-11-10T18:59:43Z] [s1] collection hw10-s1-fixed-words: 0 chunks already stored, 3,411 to embed


indexing s1:   0%|          | 0/14 [00:00<?, ?it/s]

[2026-11-10T19:16:49Z] [s1] done: 3,411 chunks in the collection | 1026.5s for 3,411 new chunks (3.3 chunks/s)
[2026-11-10T19:16:49Z] [s2] collection hw10-s2-recursive: 0 chunks already stored, 3,500 to embed


indexing s2:   0%|          | 0/14 [00:00<?, ?it/s]

[2026-11-10T19:34:08Z] [s2] done: 3,500 chunks in the collection | 1039.0s for 3,500 new chunks (3.4 chunks/s)


In [16]:
client = STORES["s1"]._client
print("collections in", CHROMA_DIR, "->", sorted(c.name for c in client.list_collections()))
for s, store in STORES.items():
    sample = store._collection.get(limit=3, include=["metadatas"])
    strategies = {m["strategy"] for m in store._collection.get(include=["metadatas"])["metadatas"]}
    print(f"{COLLECTIONS[s]}: count={store._collection.count():,} | strategies inside: {strategies} | "
          f"space={store._collection.configuration['hnsw']['space']} | sample ids: {sample['ids']}")
    assert strategies == {s}

collections in chroma_db -> ['hw10-s1-fixed-words', 'hw10-s2-recursive']


hw10-s1-fixed-words: count=3,411 | strategies inside: {'s1'} | space=cosine | sample ids: ['s1-d000-c0000', 's1-d000-c0001', 's1-d000-c0002']


hw10-s2-recursive: count=3,500 | strategies inside: {'s2'} | space=cosine | sample ids: ['s2-d000-c0000', 's2-d000-c0001', 's2-d000-c0002']


## 11. Retrieval Without Reranking

### The queries

I read the abstracts of all 100 papers and picked questions that each have one clear source paper, so I can judge relevance by hand. Most of the papers are physics, so a question about a specific method usually has one "home" paper, but similar words still show up in other papers (for example "matrix", "noise", "network"). That makes retrieval non-trivial.

- **Q1 and Q2 are the two sample queries** that I show in full detail, before and after reranking.
- Q3-Q6 are extra questions so the averages aren't based on only two queries.
- **C1-T1 and C2-T1** are the first turns of the two conversations in Section 13. On their own they are normal standalone questions, so they're part of the standalone evaluation too. The second turns (C1-T2, C2-T2) are only evaluated in the memory comparison.

In [17]:
QUERIES = {
    "Q1": "How does singular value decomposition reduce the template bank used for matched filtering of gravitational waves?",
    "Q2": "Which network topologies give high throughput for a network-on-chip turbo decoder?",
    "Q3": "How does a random phase diffuser reduce the scintillation of a laser beam?",
    "Q4": "What learning rates can regularized kernel methods achieve for additive models?",
    "Q5": "How many reticulation nodes can a binary galled network have?",
    "Q6": "How can a magnetic tunnel junction mimic short-term plasticity and long-term potentiation of a synapse?",
    "C1-T1": "How can the linewidth of an ultrahigh-Q optical cavity be measured without exciting nonlinear effects?",
    "C2-T1": "How can a transit authority estimate how many passengers travel between each pair of stations?",
}
FOLLOW_UPS = {   # second turns of the two conversations, they only make sense after the first turn
    "C1-T2": "Why does that method work better than probing the cavity with ordinary laser light?",
    "C2-T2": "Does this approach need a prior matrix to start from?",
}
STANDALONE_IDS = list(QUERIES)
SAMPLE_IDS = ["Q1", "Q2"]
for qid, q in {**QUERIES, **FOLLOW_UPS}.items():
    print(f"{qid:6} {q}")

Q1     How does singular value decomposition reduce the template bank used for matched filtering of gravitational waves?
Q2     Which network topologies give high throughput for a network-on-chip turbo decoder?
Q3     How does a random phase diffuser reduce the scintillation of a laser beam?
Q4     What learning rates can regularized kernel methods achieve for additive models?
Q5     How many reticulation nodes can a binary galled network have?
Q6     How can a magnetic tunnel junction mimic short-term plasticity and long-term potentiation of a synapse?
C1-T1  How can the linewidth of an ultrahigh-Q optical cavity be measured without exciting nonlinear effects?
C2-T1  How can a transit authority estimate how many passengers travel between each pair of stations?
C1-T2  Why does that method work better than probing the cavity with ordinary laser light?
C2-T2  Does this approach need a prior matrix to start from?


The retrieval function is the demo's `vector_store.similarity_search(query, k=3)`, but I use `similarity_search_with_score` so the cosine distance comes back too, and `k = 20` so the reranker has a candidate pool. Every run is stored in `RUNS[(query id, query variant, strategy, reranked)]`, so the metrics in Section 15 are computed from exactly the lists printed here.

In [18]:
RUNS = {}          # (qid, variant, strategy, reranked) -> ordered list of hits
QUERY_USED = {}    # (qid, variant) -> query text sent to the retriever


def retrieve(strategy, query, n=N_CANDIDATES):
    hits = STORES[strategy].similarity_search_with_score(query, k=n)
    out = []
    for rank, (doc, dist) in enumerate(hits, start=1):
        m = doc.metadata
        out.append({"rank": rank, "chunk_id": doc.id, "doc_id": m["doc_id"], "chunk_index": m["chunk_index"],
                    "source": m["source"], "distance": float(dist), "cos_sim": 1 - float(dist), "text": doc.page_content})
    return out


def show_hits(hits, k=SHOW_K, extra=(), width=330):
    cols = ["rank", "chunk_id", "doc_id", "chunk_index", "distance", "cos_sim", *extra]
    table = pd.DataFrame(hits[:k])[cols]
    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    for h in hits[:k]:
        print(f"\n  #{h['rank']} {h['chunk_id']} (paper {h['doc_id']}, {h['source']}):")
        print(textwrap.indent(textwrap.fill(h["text"][:width] + " ...", 110), "    "))


t0 = time.time()
for qid, q in QUERIES.items():
    QUERY_USED[(qid, "standalone")] = q
    for s in ["s1", "s2"]:
        RUNS[(qid, "standalone", s, False)] = retrieve(s, q)
log(f"retrieved top-{N_CANDIDATES} for {len(QUERIES)} queries x 2 strategies in {time.time() - t0:.1f}s")

[2026-11-10T19:34:11Z] retrieved top-20 for 8 queries x 2 strategies in 1.4s


In [19]:
for qid in SAMPLE_IDS:
    for s in ["s1", "s2"]:
        print("=" * 120)
        print(f"{qid} | strategy {s} | no reranking | query: {QUERIES[qid]}")
        print("=" * 120)
        show_hits(RUNS[(qid, "standalone", s, False)])
        print()

Q1 | strategy s1 | no reranking | query: How does singular value decomposition reduce the template bank used for matched filtering of gravitational waves?
 rank      chunk_id  doc_id  chunk_index  distance  cos_sim
    1 s1-d027-c0002      27            2    0.2084   0.7916
    2 s1-d027-c0013      27           13    0.2179   0.7821
    3 s1-d027-c0001      27            1    0.2289   0.7711
    4 s1-d027-c0025      27           25    0.2299   0.7701
    5 s1-d027-c0012      27           12    0.2387   0.7613

  #1 s1-d027-c0002 (paper 27, ccdv/arxiv-summarization/document/train):
    developed, either through singular-value decomposition (svd) [cite], or via a greedy algorithm [cite]. svd is
    an algebraic manipulation that transforms template waveforms into an orthonormal basis with a prescription
    that simultaneously filters out any redundancies existing within the original bank. as a result, the number of
    te ...

  #2 s1-d027-c0013 (paper 27, ccdv/arxiv-summarization/docum

## 12. Reranking

The embedding model is a **bi-encoder**: the query and the chunk are embedded separately and compared with a dot product, so it never sees them side by side. A **cross-encoder** reads `[query, chunk]` together as one input and outputs a relevance score. That's slower, but it can look at the query and chunk words together, so it's only used on the 20 candidates and not on the whole collection.

**Reranker: `cross-encoder/ms-marco-MiniLM-L6-v2`** (sentence-transformers). It's a 6-layer MiniLM (about 23M parameters) trained on MS MARCO passage ranking, which is Bing search questions paired with web passages. It's small and fast enough for my laptop (20 pairs take well under a second). One thing to keep in mind: it was trained on web search, not on physics papers full of `[math]` placeholders, so it's not obvious that it will help here. Its input limit is 512 tokens for query + chunk together, so long chunks get cut at the end. If it can't be downloaded, the notebook falls back to `ms-marco-TinyBERT-L2-v2` and logs it.

For a fair comparison the reranker gets **exactly the same 20 candidates** that ChromaDB returned for the no-reranking run. It only changes the order, so with or without reranking the same chunks are available.

In [20]:
t0 = time.time()
try:
    reranker = CrossEncoder(RERANK_MODEL, device=device)
    RERANK_USED = RERANK_MODEL
except Exception as e:
    log(f"could not load {RERANK_MODEL} ({type(e).__name__}: {e}); falling back to {RERANK_FALLBACK}")
    reranker = CrossEncoder(RERANK_FALLBACK, device=device)
    RERANK_USED = RERANK_FALLBACK
log(f"reranker {RERANK_USED} loaded in {time.time() - t0:.1f}s | max length {reranker.max_seq_length} | "
    f"parameters {sum(p.numel() for p in reranker.model.parameters()):,}")


def rerank(query, hits):
    scores = reranker.predict([(query, h["text"]) for h in hits], batch_size=16, show_progress_bar=False)
    scored = [{**h, "chroma_rank": h["rank"], "ce_score": float(sc)} for h, sc in zip(hits, scores)]
    scored.sort(key=lambda h: (-h["ce_score"], h["chroma_rank"]))   # ties keep the ChromaDB order
    for new_rank, h in enumerate(scored, start=1):
        h["rank"] = new_rank
    return scored


t0 = time.time()
for qid in QUERIES:
    for s in ["s1", "s2"]:
        RUNS[(qid, "standalone", s, True)] = rerank(QUERY_USED[(qid, "standalone")], RUNS[(qid, "standalone", s, False)])
log(f"reranked {len(QUERIES) * 2} candidate lists of {N_CANDIDATES} in {time.time() - t0:.1f}s")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[2026-11-10T19:34:12Z] reranker cross-encoder/ms-marco-MiniLM-L6-v2 loaded in 1.5s | max length 512 | parameters 22,713,601


[2026-11-10T19:34:18Z] reranked 16 candidate lists of 20 in 5.6s


### Before and after reranking for the two sample queries

The first table follows the chunks that ended up in the top 5 after reranking (plus the ones that were in the ChromaDB top 5 before) and shows where each one was before and after. The full text of the new top 5 is printed under it.

In [21]:
def before_after(qid, s, variant="standalone", k=SHOW_K):
    before = RUNS[(qid, variant, s, False)]
    after = RUNS[(qid, variant, s, True)]
    new_rank = {h["chunk_id"]: h["rank"] for h in after}
    score = {h["chunk_id"]: h["ce_score"] for h in after}
    keep = {h["chunk_id"] for h in before[:k]} | {h["chunk_id"] for h in after[:k]}
    rows = [{"chunk_id": h["chunk_id"], "doc_id": h["doc_id"], "rank before (ChromaDB)": h["rank"],
             "rank after (reranker)": new_rank[h["chunk_id"]], "cos_sim": round(h["cos_sim"], 4),
             "ce_score": round(score[h["chunk_id"]], 3)} for h in before if h["chunk_id"] in keep]
    return pd.DataFrame(rows).sort_values("rank after (reranker)")


for qid in SAMPLE_IDS:
    for s in ["s1", "s2"]:
        print("=" * 120)
        print(f"{qid} | strategy {s} | query: {QUERIES[qid]}")
        print("=" * 120)
        print(before_after(qid, s).to_string(index=False))
        print(f"\nTop {SHOW_K} after reranking:")
        show_hits(RUNS[(qid, "standalone", s, True)], extra=("chroma_rank", "ce_score"), width=260)
        print()

Q1 | strategy s1 | query: How does singular value decomposition reduce the template bank used for matched filtering of gravitational waves?
     chunk_id  doc_id  rank before (ChromaDB)  rank after (reranker)  cos_sim  ce_score
s1-d027-c0002      27                       1                      1   0.7916     5.882
s1-d027-c0001      27                       3                      2   0.7711     0.984
s1-d027-c0013      27                       2                      3   0.7821     0.818
s1-d027-c0000      27                      10                      4   0.7480     0.190
s1-d027-c0031      27                       6                      5   0.7574     0.173
s1-d027-c0025      27                       4                      7   0.7701    -1.115
s1-d027-c0012      27                       5                     10   0.7613    -2.187

Top 5 after reranking:
 rank      chunk_id  doc_id  chunk_index  distance  cos_sim  chroma_rank  ce_score
    1 s1-d027-c0002      27            2    0.208

## 13. Conversational Memory

### How the demo does it

In `Demo_10_RAG_Chatbot_Local.ipynb` the memory is just the `messages` list. Every `HumanMessage` and `AIMessage` is appended, and the whole list is sent to `chat.invoke(messages)`, so the LLM sees the earlier turns. The demo also imports `InMemoryChatMessageHistory` and `RunnableWithMessageHistory` for this but doesn't use them.

There's one problem for RAG, though. In the demo, `augment_prompt(query)` searches the vector store with **only the newest question**. The LLM has the history, but the retriever doesn't. So if the second question is *"Why does that method work better ...?"*, the retriever searches for "that method" and has no idea which method that is.

### What I implemented

I kept it simple and close to the demo:

1. **Chat history.** One `InMemoryChatMessageHistory` per conversation (`session_id`), and the answer chain is wrapped in `RunnableWithMessageHistory`. This is the same "append every message" idea as the demo's list, done with the classes the demo imports. Only the user's question and the LLM's answer are stored. The retrieved chunks aren't, so the history stays short.
2. **The retriever also uses the memory.** I compare three versions of the follow-up query:
   - **No memory:** the follow-up is sent to ChromaDB as it is, like the demo.
   - **Memory (concat):** the previous user question is glued in front of the follow-up. No LLM is involved, so this is the simplest possible memory and it's fully deterministic.
   - **Memory (LLM rewrite):** `llama3.2:3b` gets the chat history and rewrites the follow-up into a standalone question ("that method" becomes "injecting squeezed vacuum ..."), and that rewrite is what gets searched. `temperature=0` and `seed=SEED`, so it gives the same rewrite every run.
3. **Answers.** The LLM answers with the demo's prompt idea ("use the contexts below"), using the top 3 reranked chunks from strategy 2. The answers are only there to show the chatbot working. I don't score them.

No agent framework. The rewrite step is one prompt.

One small note: LangChain 1.7 marks `RunnableWithMessageHistory` and `InMemoryChatMessageHistory` as deprecated and recommends LangGraph persistence instead. I kept them anyway. They're what the demo uses, they still work, and switching would mean adding LangGraph, which is more framework than this simple memory needs. I turned the deprecation warning off in Section 3 so it doesn't clutter the output.

In [22]:
chat = ChatOllama(model=LLM_MODEL, temperature=0, seed=SEED, num_predict=220, num_ctx=4096)
try:
    t0 = time.time()
    chat.invoke([HumanMessage(content="Reply with the single word OK.")])
    LLM_OK = True
    log(f"Ollama model {LLM_MODEL} is reachable ({time.time() - t0:.1f}s for a test message)")
except Exception as e:
    LLM_OK = False
    log(f"!! Ollama model {LLM_MODEL} is NOT reachable ({type(e).__name__}: {e}). The LLM rewrite and the answers "
        f"will be skipped and marked as 'not run'. The no-memory and concat-memory retrieval still run.")

# memory store, one history per session (same pattern the demo imports)
SESSION_STORE = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in SESSION_STORE:
        SESSION_STORE[session_id] = InMemoryChatMessageHistory()
    return SESSION_STORE[session_id]


# answer prompt: the demo's augment_prompt, plus a placeholder for the history
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Answer the question using the contexts from arXiv papers. "
               "If the contexts do not contain the answer, say so. Keep the answer to 3-4 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "Using the contexts below, answer the query.\n\nContexts:\n{context}\n\nQuery: {question}"),
])
answer_chain = answer_prompt | chat
chat_with_memory = RunnableWithMessageHistory(
    answer_chain, get_session_history, input_messages_key="question", history_messages_key="history")

# rewrite prompt: turns a follow-up into a standalone search query using the history
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You rewrite follow-up questions for a search engine. You never answer questions."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "Rewrite my next question so it can be understood without the conversation above. "
              "Replace words like 'that method', 'this approach', 'it' with the specific thing they refer to. "
              "Keep it a question, keep it short, and do not answer it. Output only the rewritten question.\n\n"
              "Next question: {question}"),
])
rewrite_chain = rewrite_prompt | chat


def make_context(hits, n=N_CONTEXT):
    return "\n\n".join(f"[Context {i} | paper {h['doc_id']}] {h['text']}" for i, h in enumerate(hits[:n], 1))


def concat_memory_query(session_id, follow_up):
    previous_user = [m.content for m in get_session_history(session_id).messages if isinstance(m, HumanMessage)]
    return (previous_user[-1] + " " + follow_up) if previous_user else follow_up


def run_retrieval(qid, variant, query):
    QUERY_USED[(qid, variant)] = query
    for s in ["s1", "s2"]:
        RUNS[(qid, variant, s, False)] = retrieve(s, query)
        RUNS[(qid, variant, s, True)] = rerank(query, RUNS[(qid, variant, s, False)])


def short(hits, k=3):
    return pd.DataFrame(hits[:k])[["rank", "chunk_id", "doc_id", "cos_sim"] + (["ce_score"] if "ce_score" in hits[0] else [])]


ANSWERS = {}
REWRITES = {}

[2026-11-10T19:34:25Z] Ollama model llama3.2:3b is reachable (6.5s for a test message)


### Running the two conversations

For each conversation:

1. **Turn 1** is retrieved (top 3 reranked chunks from strategy 2) and answered through `chat_with_memory`, which saves the question and the answer in the session history.
2. **Turn 2 without memory**: the raw follow-up goes to the retriever, and the LLM answers with an empty history. This is what the demo's `augment_prompt` would do with a fresh chat.
3. **Turn 2 with memory**: the concat query and the LLM rewrite are both built from the stored history. The LLM rewrite is used for the chatbot answer, which goes through `chat_with_memory` again, so the history now has 4 messages.

In [23]:
CONVERSATIONS = {"C1": ("C1-T1", "C1-T2"), "C2": ("C2-T1", "C2-T2")}
t_conv = time.time()
for conv, (t1, t2) in CONVERSATIONS.items():
    session = f"{conv}-memory"
    print("#" * 120)
    print(f"Conversation {conv}")
    print("#" * 120)

    # ---- turn 1 (already retrieved and reranked in Sections 11-12)
    q1 = QUERIES[t1]
    hits1 = RUNS[(t1, "standalone", CHAT_STRATEGY, True)]
    print(f"\nUSER (turn 1): {q1}")
    print(short(hits1).to_string(index=False))
    if LLM_OK:
        ans = chat_with_memory.invoke({"question": q1, "context": make_context(hits1)},
                                      config={"configurable": {"session_id": session}})
        ANSWERS[(t1, "memory")] = ans.content
        print("ASSISTANT:", textwrap.fill(ans.content, 110, subsequent_indent="           "))
    else:
        get_session_history(session).add_user_message(q1)   # keep the user turn so the concat memory still works
        print("ASSISTANT: (not run, Ollama unavailable)")

    # ---- turn 2 without memory
    q2 = FOLLOW_UPS[t2]
    run_retrieval(t2, "no_memory", q2)
    hits_nm = RUNS[(t2, "no_memory", CHAT_STRATEGY, True)]
    print(f"\nUSER (turn 2): {q2}")
    print(f"\n--- WITHOUT memory | retriever query: {q2!r}")
    print(short(hits_nm).to_string(index=False))
    if LLM_OK:
        ans = answer_chain.invoke({"history": [], "question": q2, "context": make_context(hits_nm)})
        ANSWERS[(t2, "no_memory")] = ans.content
        print("ASSISTANT:", textwrap.fill(ans.content, 110, subsequent_indent="           "))

    # ---- turn 2 with memory (concat)
    q_concat = concat_memory_query(session, q2)
    run_retrieval(t2, "memory_concat", q_concat)
    print(f"\n--- WITH memory (concat) | retriever query: {q_concat!r}")
    print(short(RUNS[(t2, "memory_concat", CHAT_STRATEGY, True)]).to_string(index=False))

    # ---- turn 2 with memory (LLM rewrite) + answer with history
    if LLM_OK:
        rewritten = rewrite_chain.invoke({"history": get_session_history(session).messages, "question": q2}).content
        rewritten = rewritten.strip().strip('"').splitlines()[0].strip()
        REWRITES[t2] = rewritten
        run_retrieval(t2, "memory_llm", rewritten)
        hits_m = RUNS[(t2, "memory_llm", CHAT_STRATEGY, True)]
        print(f"\n--- WITH memory (LLM rewrite) | retriever query: {rewritten!r}")
        print(short(hits_m).to_string(index=False))
        ans = chat_with_memory.invoke({"question": q2, "context": make_context(hits_m)},
                                      config={"configurable": {"session_id": session}})
        ANSWERS[(t2, "memory_llm")] = ans.content
        print("ASSISTANT:", textwrap.fill(ans.content, 110, subsequent_indent="           "))
    else:
        print("\n--- WITH memory (LLM rewrite): not run, Ollama unavailable")
    print()
log(f"conversations finished in {time.time() - t_conv:.1f}s")

########################################################################################################################
Conversation C1
########################################################################################################################

USER (turn 1): How can the linewidth of an ultrahigh-Q optical cavity be measured without exciting nonlinear effects?
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d033-c0002      33 0.848977  5.349281
    2 s2-d033-c0019      33 0.778284  3.569654
    3 s2-d033-c0018      33 0.877099  3.564631


ASSISTANT: According to the contexts, the linewidth of an ultrahigh-Q optical cavity can be measured without exciting
           nonlinear effects by injecting a squeezed vacuum field into a detuned cavity. The linewidth is then
           measured by observing the destruction of the correlation between the upper and lower quantum
           sidebands with respect to the carrier frequency. This method is advantageous over traditional
           optical methods as it does not excite any nonlinear processes in the cavity.



USER (turn 2): Why does that method work better than probing the cavity with ordinary laser light?

--- WITHOUT memory | retriever query: 'Why does that method work better than probing the cavity with ordinary laser light?'
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d033-c0002      33 0.745678 -2.263739
    2 s2-d033-c0015      33 0.715053 -2.562834
    3 s2-d033-c0001      33 0.734793 -2.636981


ASSISTANT: According to the contexts, the method works better than probing the cavity with ordinary laser light because
           it utilizes the injection of squeezed vacuum into a test cavity, which transmits only the upper or
           lower quantum sidebands within the cavity linewidth while reflecting the counterparts, allowing for
           the measurement of the cavity's linewidth and q factor without exciting any nonlinear processes in
           the cavity.



--- WITH memory (concat) | retriever query: 'How can the linewidth of an ultrahigh-Q optical cavity be measured without exciting nonlinear effects? Why does that method work better than probing the cavity with ordinary laser light?'
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d033-c0002      33 0.846697  5.722953
    2 s2-d033-c0018      33 0.859446  4.076773
    3 s2-d033-c0019      33 0.768193  3.007894



--- WITH memory (LLM rewrite) | retriever query: 'Why does injecting a squeezed vacuum field into a detuned cavity work better than probing the cavity with ordinary laser light?'
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d033-c0019      33 0.808253  4.836731
    2 s2-d033-c0002      33 0.776069  3.891221
    3 s2-d033-c0016      33 0.785692  2.624945


... (4 outputs left out)

ASSISTANT: Based on the contexts provided, it appears that the approach does not necessarily require a prior matrix to
           start from. In fact, the authors mention that they use an ad hoc estimation of the origin-
           destination (o-d) matrix, which relies on the eigenvalues of the survey matrix, rather than relying
           on a prior matrix. This approach is used to overcome the assumption of equal behavior between car-
           user and public transport users, and to account for the potential biases in the survey results.



--- WITH memory (concat) | retriever query: 'How can a transit authority estimate how many passengers travel between each pair of stations? Does this approach need a prior matrix to start from?'
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d032-c0001      32 0.842012  2.036262
    2 s2-d032-c0000      32 0.797486  0.479810
    3 s2-d032-c0005      32 0.839015 -0.630806



--- WITH memory (LLM rewrite) | retriever query: 'Does the gravity-type model require a prior origin-destination matrix to estimate the number of passengers traveling between each pair of stations?'
 rank      chunk_id  doc_id  cos_sim  ce_score
    1 s2-d032-c0001      32 0.811129  3.334517
    2 s2-d032-c0000      32 0.771856  1.973307
    3 s2-d032-c0003      32 0.797600  1.744365


ASSISTANT: According to the contexts, the approach to estimating the origin-destination (OD) matrix, such as the gravity-
           type model, does not necessarily require a prior matrix to start from. Instead, it uses incomplete
           traffic counts and other available information to introduce prior knowledge on traffic flows and
           assign a cost to each journey. The prior knowledge is often based on general assumptions about
           traffic patterns, but it is not necessarily a pre-existing OD matrix.

[2026-11-10T19:36:47Z] conversations finished in 142.1s


In [24]:
print("What is stored in memory at the end (session histories):\n")
for session, hist in SESSION_STORE.items():
    print(f"[{session}] {len(hist.messages)} messages")
    for m in hist.messages:
        print(f"   {type(m).__name__:12} {m.content[:150]!r}{' ...' if len(m.content) > 150 else ''}")
print("\nQuery sent to the retriever for each follow-up variant:")
print(pd.DataFrame([{"follow-up": qid, "variant": v, "retriever query": q}
                    for (qid, v), q in QUERY_USED.items() if qid in FOLLOW_UPS]).to_string(index=False))

What is stored in memory at the end (session histories):

[C1-memory] 4 messages
   HumanMessage 'How can the linewidth of an ultrahigh-Q optical cavity be measured without exciting nonlinear effects?'
   AIMessage    'According to the contexts, the linewidth of an ultrahigh-Q optical cavity can be measured without exciting nonlinear effects by injecting a squeezed v' ...
   HumanMessage 'Why does that method work better than probing the cavity with ordinary laser light?'
   AIMessage    'According to Context 2, the method works better than probing the cavity with ordinary laser light because it utilizes the injection of squeezed vacuum' ...
[C2-memory] 4 messages
   HumanMessage 'How can a transit authority estimate how many passengers travel between each pair of stations?'
   AIMessage    'Based on the contexts provided, a transit authority such as CityRail and Co can estimate the number of passengers traveling between each pair of stati' ...
   HumanMessage 'Does this approach need 

## 14. Manual Relevance Labels

There are thousands of chunks, so I couldn't label every chunk for every query by hand. I also couldn't label just the retrieved chunks, because the two strategies cut the papers in different places and label sets from one strategy wouldn't carry over to the other. So I did it like this:

1. **I read the target paper for each question** (the abstract and the parts about the question) and wrote down the 3-4 **facts a good answer needs**. For example, for Q2 those are: generalized de Bruijn and Kautz topologies are best, there's a throughput vs. area/complexity trade-off, and the parallelism degree, injection rate and routing algorithm also matter.
2. For each fact I wrote down the **key phrase(s)** that show up whenever the paper talks about it (for example `de-bruijn|kautz`). I checked these against the paper text, and the check below shows that every phrase really occurs in its target paper.
3. **A retrieved chunk counts as relevant if it comes from the target paper AND contains at least one of the facts.** So a chunk from the right paper that is only the reference list or an unrelated derivation doesn't count. A chunk from another paper doesn't count either, even if it has similar words, because it can't answer a question about this paper's method.

I apply the same rules to both strategies, with and without reranking and with or without memory, so no method gets judged more generously than another. Having the rules in code also means anyone can check them, and the tables in Section 15 print the label of every top-ranked chunk.

**Follow-up questions.** C1-T2 and C2-T2 are judged by **what the user actually meant**, i.e. the question after "that method" is resolved. For C1-T2 that means *why squeezed-vacuum probing is better than a laser probe* (paper 33). The exact same labels are used for the no-memory query and for both memory queries. A chunk that matches the literal words "that method work better" in some unrelated paper doesn't count, because it doesn't answer what the user asked.

In [25]:
RELEVANCE = {
    "Q1": dict(doc=27, facts={
        "SVD turns the template bank into a smaller orthonormal (reduced) basis": r"orthonormal basis|reduced basis|reduced svd basis",
        "fewer templates / basis waveforms are needed for matched filtering": r"number of templates|templates required|number of basis waveforms",
        "the SVD basis removes the mass bias of the phenomenological templates (better match to NR)": r"mass-bias|mass bias|accuracy booster|faithfulness"}),
    "Q2": dict(doc=99, facts={
        "generalized de Bruijn and generalized Kautz topologies perform best": r"de-bruijn|kautz",
        "the trade-off is throughput versus area / complexity overhead": r"complexity overhead|area overhead",
        "parallelism degree, message injection rate and routing algorithm also matter": r"parallelism degree|injection rate|routing algorithm"}),
    "Q3": dict(doc=3, facts={
        "the diffuser adds a random phase and makes the beam partially coherent": r"partially coherent|random phase|additional phase",
        "the scintillation index / intensity fluctuations go down": r"suppress\w*|decrease in|reduc\w* (of )?(the )?(scintillation|intensity fluctuations)",
        "finite-time phase variations and the detector integration time limit the effect": r"finite-time|integration time|response time"}),
    "Q4": dict(doc=0, facts={
        "the paper derives learning rates for SVMs / kernel methods with additive kernels": r"learning rates? (for|of)",
        "additive models do better in high dimensions (less curse of dimensionality)": r"high dimension|curse of",
        "quantile regression with the pinball loss is the main example": r"quantile regression|pinball",
        "the rates are compared with the multivariate Gaussian RBF kernel": r"gaussian (rbf )?kernel|radial basis|multivariate gaussian"}),
    "Q5": dict(doc=58, facts={
        "a galled network with n leaves has a (tight) upper bound on reticulation nodes": r"reticulations? (nodes )?in (a )?galled network|galled networks? with",
        "galled networks are reticulation visible (level-1) networks": r"galled networks (are|form)",
        "related bounds are proved for nearly-stable and stable-child networks": r"(nearly-stable|stable-child) networks? with"}),
    "Q6": dict(doc=7, facts={
        "short-term plasticity is a meta-stable state that decays after the stimulus": r"short-term plasticity|meta-stable|\bstp\b",
        "long-term potentiation happens after crossing the energy barrier": r"long-term potentiation|energy barrier|\bltp\b",
        "spin-polarized current switches the free layer (like neurotransmitter release)": r"spin-polariz|neurotransmitter",
        "the frequency of the input stimulus decides whether LTP happens": r"frequency of the (input|stimulus)"}),
    "C1-T1": dict(doc=33, facts={
        "inject a squeezed vacuum field into the cavity": r"squeezed vacuum",
        "the linewidth shows up as destroyed correlation between upper and lower sidebands": r"upper and lower (quantum )?sidebands|destr\w+ (of )?(the )?correlation",
        "the carrier is detuned and the reflected field is measured with homodyne detection": r"detuned|homodyne"}),
    "C1-T2": dict(doc=33, facts={
        "squeezed vacuum does not excite nonlinear processes (good for ultrahigh-Q / WGM cavities)": r"nonlinear process",
        "classical probe light makes the Q measurement power-dependent and unreliable": r"power-dependent|unreliable|light-induced",
        "squeezing improves the signal contrast beyond the shot-noise limit": r"signal contrast|shot noise|shot-noise"}),
    "C2-T1": dict(doc=32, facts={
        "estimate the origin-destination (OD) matrix": r"origin-destination|od matrix",
        "use ticket records / passenger counts at the stations": r"ticket|boom gate|counts",
        "use survey data collected from passengers": r"survey"}),
    "C2-T2": dict(doc=32, facts={
        "other methods rely on a prior (a priori) OD matrix": r"\bprior\b|a priori",
        "this paper estimates the matrix structure from survey data through its eigenvectors instead": r"eigenvector",
        "the estimator is made robust to biased surveys": r"robust"}),
}
assert set(RELEVANCE) == set(QUERIES) | set(FOLLOW_UPS)


def judge(qid, hit):
    # returns (is_relevant, list of facts the chunk covers)
    rule = RELEVANCE[qid]
    if hit["doc_id"] != rule["doc"]:
        return False, []
    facts = [f for f, pattern in rule["facts"].items() if re.search(pattern, hit["text"])]
    return len(facts) > 0, facts


rows = []
for qid, rule in RELEVANCE.items():
    paper = docs[rule["doc"]]["text"]
    for fact, pattern in rule["facts"].items():
        n_rel = {s: sum(judge(qid, c)[1].count(fact) for c in CHUNKS[s] if c["doc_id"] == rule["doc"]) for s in ["s1", "s2"]}
        rows.append({"query": qid, "paper": rule["doc"], "fact": fact[:70], "phrase hits in paper": len(re.findall(pattern, paper)),
                     "s1 chunks with fact": n_rel["s1"], "s2 chunks with fact": n_rel["s2"]})
fact_table = pd.DataFrame(rows)
print(fact_table.to_string(index=False))
if not SMOKE_TEST:
    assert (fact_table["phrase hits in paper"] > 0).all(), "a relevance phrase never occurs in its target paper"

query  paper                                                                   fact  phrase hits in paper  s1 chunks with fact  s2 chunks with fact
   Q1     27 SVD turns the template bank into a smaller orthonormal (reduced) basis                    25                   16                   17
   Q1     27     fewer templates / basis waveforms are needed for matched filtering                     4                    4                    4
   Q1     27 the SVD basis removes the mass bias of the phenomenological templates                     12                    7                    8
   Q2     99    generalized de Bruijn and generalized Kautz topologies perform best                    10                    7                    6
   Q2     99          the trade-off is throughput versus area / complexity overhead                     5                    3                    3
   Q2     99 parallelism degree, message injection rate and routing algorithm also                     17       

In [26]:
# total number of relevant chunks per query in each collection (needed for the chunk-level recall)
TOTAL_RELEVANT = {(qid, s): sum(judge(qid, c)[0] for c in CHUNKS[s]) for qid in RELEVANCE for s in ["s1", "s2"]}
summary = pd.DataFrame([{"query": qid, "target paper": rule["doc"], "facts": len(rule["facts"]),
                         "relevant chunks s1": TOTAL_RELEVANT[(qid, "s1")],
                         "chunks of paper s1": sum(c["doc_id"] == rule["doc"] for c in chunks_s1),
                         "relevant chunks s2": TOTAL_RELEVANT[(qid, "s2")],
                         "chunks of paper s2": sum(c["doc_id"] == rule["doc"] for c in chunks_s2)}
                        for qid, rule in RELEVANCE.items()])
print(summary.to_string(index=False))

query  target paper  facts  relevant chunks s1  chunks of paper s1  relevant chunks s2  chunks of paper s2
   Q1            27      3                  19                  34                  21                  36
   Q2            99      3                   8                  12                   7                  12
   Q3             3      3                  12                  19                  10                  20
   Q4             0      4                  19                  24                  18                  24
   Q5            58      3                   8                  63                   8                  57
   Q6             7      4                  12                  18                  14                  19
C1-T1            33      3                  17                  19                  16                  20
C1-T2            33      3                   9                  19                  10                  20
C2-T1            32      3           

## 15. Evaluation Metrics

For each ranked list (one query, one strategy, with or without reranking, and for follow-ups one memory variant), with K = 3 and K = 5:

- **Precision@K** = (relevant chunks in the top K) / K. It shows how much of what we'd give the LLM is actually useful.
- **Recall@K** = (facts covered by the relevant chunks in the top K) / (number of facts for that query). In other words: how much of the answer is in the top K. I count **facts** instead of chunks on purpose. Strategy 1 and strategy 2 cut a paper into a different number of chunks, so "relevant chunks found / all relevant chunks" would be measured against different totals for the two strategies, and in long papers it's tiny for any K. Facts are the same no matter how the paper is cut. I still report the usual chunk-level recall in one extra table for completeness.
- **MRR@10** = 1 / (rank of the first relevant chunk), or 0 if there's none in the top 10, averaged over queries. It shows how quickly the first useful chunk shows up.

All the numbers below are computed from the `RUNS` lists filled in Sections 11-13. The small test checks the metric functions on a made-up example first.

In [27]:
def precision_at_k(labels, k):
    return sum(labels[:k]) / k


def fact_recall_at_k(fact_lists, k, n_facts):
    covered = set()
    for facts in fact_lists[:k]:
        covered.update(facts)
    return len(covered) / n_facts


def chunk_recall_at_k(labels, k, total_relevant):
    return sum(labels[:k]) / total_relevant if total_relevant else float("nan")


def reciprocal_rank(labels, depth=MRR_DEPTH):
    for i, rel in enumerate(labels[:depth], start=1):
        if rel:
            return 1 / i
    return 0.0


# sanity check on a toy ranking: relevant at ranks 2 and 3, facts A and B (out of A, B, C)
toy_labels = [False, True, True, False, False]
toy_facts = [[], ["A"], ["A", "B"], [], []]
assert precision_at_k(toy_labels, 3) == 2 / 3 and precision_at_k(toy_labels, 5) == 2 / 5
assert fact_recall_at_k(toy_facts, 2, 3) == 1 / 3 and fact_recall_at_k(toy_facts, 5, 3) == 2 / 3
assert reciprocal_rank(toy_labels) == 0.5 and reciprocal_rank([False] * 5) == 0.0
assert chunk_recall_at_k(toy_labels, 5, 4) == 0.5
print("metric functions pass the toy test")


def evaluate(key):
    qid, variant, s, reranked = key
    judged = [judge(qid, h) for h in RUNS[key]]
    labels = [r for r, _ in judged]
    facts = [f for _, f in judged]
    row = {"query": qid, "variant": variant, "strategy": s, "reranked": reranked}
    for k in K_VALUES:
        row[f"P@{k}"] = precision_at_k(labels, k)
        row[f"R@{k}"] = fact_recall_at_k(facts, k, len(RELEVANCE[qid]["facts"]))
    row[f"MRR@{MRR_DEPTH}"] = reciprocal_rank(labels)
    for k in K_VALUES:
        row[f"chunkR@{k}"] = chunk_recall_at_k(labels, k, TOTAL_RELEVANT[(qid, s)])
    row["top10 labels"] = "".join("R" if r else "." for r in labels[:10])
    row["top5 papers"] = " ".join(str(h["doc_id"]) for h in RUNS[key][:5])
    return row


results = pd.DataFrame([evaluate(key) for key in RUNS])
METRIC_COLS = [f"P@{k}" for k in K_VALUES] + [f"R@{k}" for k in K_VALUES] + [f"MRR@{MRR_DEPTH}"]
log(f"evaluated {len(results)} ranked lists ({results.variant.value_counts().to_dict()})")

metric functions pass the toy test
[2026-11-10T19:36:48Z] evaluated 56 ranked lists ({'standalone': 32, 'no_memory': 8, 'memory_concat': 8, 'memory_llm': 8})


### Checking the labels against the real chunks

Here is every standalone ranked list in one line: `top10 labels` marks a relevant chunk with `R` and a non-relevant one with `.`, in rank order, and `top5 papers` lists the paper ids of the top 5 chunks. So for every metric you can see which chunks it comes from (P@5 is just the number of `R`s in the first 5 characters divided by 5). Below that, the top 5 of the two sample queries are printed with the facts each chunk was matched on.

In [28]:
view = results[results.variant == "standalone"][["query", "strategy", "reranked", "top10 labels", "top5 papers"] + METRIC_COLS]
print(view.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

query strategy  reranked top10 labels    top5 papers   P@3   P@5   R@3   R@5  MRR@10
   Q1       s1     False   RRR.RR..R. 27 27 27 27 27 1.000 0.800 0.667 0.667   1.000
   Q1       s2     False   RRR.RRRRRR 27 27 27 27 27 1.000 0.800 0.667 0.667   1.000
   Q2       s1     False   RRRRR...RR 99 99 99 99 99 1.000 1.000 1.000 1.000   1.000
   Q2       s2     False   RRRR....RR 99 99 99 99 99 1.000 0.800 0.667 1.000   1.000
   Q3       s1     False   RRRRRRRRRR      3 3 3 3 3 1.000 1.000 1.000 1.000   1.000
   Q3       s2     False   RRRRRRRRR.      3 3 3 3 3 1.000 1.000 1.000 1.000   1.000
   Q4       s1     False   RRRRRRRRRR      0 0 0 0 0 1.000 1.000 1.000 1.000   1.000
   Q4       s2     False   RRRRRRRR.R      0 0 0 0 0 1.000 1.000 1.000 1.000   1.000
   Q5       s1     False   R...RRRRR. 58 58 58 58 58 0.333 0.400 0.667 1.000   1.000
   Q5       s2     False   RRR.R.R... 58 58 58 58 58 1.000 0.800 0.667 1.000   1.000
   Q6       s1     False   .RRRRR..RR      7 7 7 7 7 0.667 0.800 

In [29]:
for qid in SAMPLE_IDS:
    for s in ["s1", "s2"]:
        for rr in [False, True]:
            print(f"--- {qid} | {s} | {'reranked' if rr else 'no rerank'} | target paper {RELEVANCE[qid]['doc']}")
            for h in RUNS[(qid, "standalone", s, rr)][:SHOW_K]:
                rel, facts = judge(qid, h)
                print(f"  #{h['rank']} {h['chunk_id']:16} paper {h['doc_id']:>2}  {'RELEVANT' if rel else 'not relevant':12} "
                      f"facts: {[f[:45] for f in facts]}")

--- Q1 | s1 | no rerank | target paper 27
  #1 s1-d027-c0002    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or', 'fewer templates / basis waveforms are needed ']
  #2 s1-d027-c0013    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or']
  #3 s1-d027-c0001    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or', 'fewer templates / basis waveforms are needed ']
  #4 s1-d027-c0025    paper 27  not relevant facts: []
  #5 s1-d027-c0012    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or', 'fewer templates / basis waveforms are needed ']
--- Q1 | s1 | reranked | target paper 27
  #1 s1-d027-c0002    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or', 'fewer templates / basis waveforms are needed ']
  #2 s1-d027-c0001    paper 27  RELEVANT     facts: ['SVD turns the template bank into a smaller or', 'fewer templates / basis waveforms are needed ']
  #3 

## 16. Results Tables

### (a) All four retrieval conditions (mean over the 8 standalone queries)

In [30]:
std = results[results.variant == "standalone"].copy()
std["condition"] = std.strategy.map({"s1": "S1 fixed words", "s2": "S2 recursive sentences"}) + \
    std.reranked.map({False: " | no rerank", True: " | rerank"})
table_a = std.groupby("condition")[METRIC_COLS].mean()
print(table_a.to_string(float_format=lambda x: f"{x:.3f}"))

                                     P@3   P@5   R@3   R@5  MRR@10
condition                                                         
S1 fixed words | no rerank         0.833 0.850 0.885 0.958   0.938
S1 fixed words | rerank            0.875 0.825 0.917 0.958   0.917
S2 recursive sentences | no rerank 0.917 0.850 0.844 0.958   1.000
S2 recursive sentences | rerank    0.833 0.800 0.885 0.958   0.938


### (b) Per-query results (P@5 / R@5 / MRR@10)

In [31]:
per_query = std.pivot_table(index="query", columns="condition", values=["P@5", "R@5", f"MRR@{MRR_DEPTH}"])
per_query = per_query.reindex(STANDALONE_IDS)
for metric in ["P@5", "R@5", f"MRR@{MRR_DEPTH}"]:
    print(f"\n{metric}")
    print(per_query[metric].to_string(float_format=lambda x: f"{x:.3f}"))


P@5
condition  S1 fixed words | no rerank  S1 fixed words | rerank  S2 recursive sentences | no rerank  S2 recursive sentences | rerank
query                                                                                                                              
Q1                              0.800                    0.800                               0.800                            0.800
Q2                              1.000                    0.600                               0.800                            0.600
Q3                              1.000                    0.800                               1.000                            1.000
Q4                              1.000                    1.000                               1.000                            1.000
Q5                              0.400                    0.400                               0.800                            0.400
Q6                              0.800                    1.000         

### (c) Chunking strategy 1 vs. strategy 2

In [32]:
rows = []
for rr in [False, True]:
    sub = std[std.reranked == rr]
    m1 = sub[sub.strategy == "s1"][METRIC_COLS].mean()
    m2 = sub[sub.strategy == "s2"][METRIC_COLS].mean()
    tag = "rerank" if rr else "no rerank"
    rows += [{"setting": tag, "row": "S1 fixed words", **m1}, {"setting": tag, "row": "S2 recursive", **m2},
             {"setting": tag, "row": "S2 - S1", **(m2 - m1)}]
table_c = pd.DataFrame(rows).set_index(["setting", "row"])
print(table_c.to_string(float_format=lambda x: f"{x:+.3f}" if x < 0 else f"{x:.3f}"))

wins = std.pivot_table(index=["query", "reranked"], columns="strategy", values=f"MRR@{MRR_DEPTH}")
print(f"\nMRR@{MRR_DEPTH} per query: S2 better in {(wins.s2 > wins.s1).sum()}, same in {(wins.s2 == wins.s1).sum()}, "
      f"S1 better in {(wins.s2 < wins.s1).sum()} of {len(wins)} (query, rerank) pairs")

                            P@3    P@5    R@3   R@5  MRR@10
setting   row                                              
no rerank S1 fixed words  0.833  0.850  0.885 0.958   0.938
          S2 recursive    0.917  0.850  0.844 0.958   1.000
          S2 - S1         0.083  0.000 -0.042 0.000   0.062
rerank    S1 fixed words  0.875  0.825  0.917 0.958   0.917
          S2 recursive    0.833  0.800  0.885 0.958   0.938
          S2 - S1        -0.042 -0.025 -0.031 0.000   0.021

MRR@10 per query: S2 better in 2, same in 14, S1 better in 0 of 16 (query, rerank) pairs


### (d) Without reranking vs. with reranking

In [33]:
rows = []
for s in ["s1", "s2"]:
    sub = std[std.strategy == s]
    m0 = sub[~sub.reranked][METRIC_COLS].mean()
    m1 = sub[sub.reranked][METRIC_COLS].mean()
    rows += [{"strategy": s, "row": "no rerank", **m0}, {"strategy": s, "row": "rerank", **m1},
             {"strategy": s, "row": "rerank - no rerank", **(m1 - m0)}]
table_d = pd.DataFrame(rows).set_index(["strategy", "row"])
print(table_d.to_string(float_format=lambda x: f"{x:+.3f}" if x < 0 else f"{x:.3f}"))

paired = std.pivot_table(index=["query", "strategy"], columns="reranked", values=["P@5", f"MRR@{MRR_DEPTH}"])
for metric in ["P@5", f"MRR@{MRR_DEPTH}"]:
    d = paired[metric][True] - paired[metric][False]
    print(f"{metric}: reranking better in {(d > 0).sum()}, same in {(d == 0).sum()}, worse in {(d < 0).sum()} "
          f"of {len(d)} (query, strategy) pairs")

                               P@3    P@5   R@3   R@5  MRR@10
strategy row                                                 
s1       no rerank           0.833  0.850 0.885 0.958   0.938
         rerank              0.875  0.825 0.917 0.958   0.917
         rerank - no rerank  0.042 -0.025 0.031 0.000  -0.021
s2       no rerank           0.917  0.850 0.844 0.958   1.000
         rerank              0.833  0.800 0.885 0.958   0.938
         rerank - no rerank -0.083 -0.050 0.042 0.000  -0.062
P@5: reranking better in 3, same in 9, worse in 4 of 16 (query, strategy) pairs
MRR@10: reranking better in 1, same in 13, worse in 2 of 16 (query, strategy) pairs


### (e) Without memory vs. with memory (follow-up questions C1-T2 and C2-T2)

In [34]:
fu = results[results["query"].isin(list(FOLLOW_UPS))].copy()
variant_order = {"no_memory": 0, "memory_concat": 1, "memory_llm": 2}
fu["v"] = fu.variant.map(variant_order)
table_e = fu.sort_values(["query", "strategy", "reranked", "v"])[
    ["query", "strategy", "reranked", "variant", "top10 labels", "top5 papers"] + METRIC_COLS]
print(table_e.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nMean over both follow-ups, both strategies and both rerank settings:")
table_e_mean = fu.groupby("variant")[METRIC_COLS].mean().reindex([v for v in variant_order if v in set(fu.variant)])
print(table_e_mean.to_string(float_format=lambda x: f"{x:.3f}"))
for variant in variant_order:
    if variant not in set(fu.variant):
        print(f"{variant}: NOT RUN (Ollama was not available)")

query strategy  reranked       variant top10 labels    top5 papers   P@3   P@5   R@3   R@5  MRR@10
C1-T2       s1     False     no_memory   R.R.RRR.R. 33 33 33 33 33 0.667 0.600 0.667 1.000   1.000
C1-T2       s1     False memory_concat   R.RR.RRR.. 33 33 33 33 33 0.667 0.600 0.667 0.667   1.000
C1-T2       s1     False    memory_llm   .RR.RRR... 33 33 33 33 33 0.667 0.600 1.000 1.000   0.500
C1-T2       s1      True     no_memory   R.R..RR.R. 33 33 33 33 33 0.667 0.400 0.667 0.667   1.000
C1-T2       s1      True memory_concat   .RRRRRR... 33 33 33 33 33 0.667 0.800 0.667 1.000   0.500
C1-T2       s1      True    memory_llm   R.R..RR.R. 33 33 33 33 33 0.667 0.400 1.000 1.000   1.000
C1-T2       s2     False     no_memory   .RRR..R... 33 33 33 33 33 0.667 0.600 0.333 0.667   0.500
C1-T2       s2     False memory_concat   RRRRR..RR. 33 33 33 33 33 1.000 1.000 0.667 1.000   1.000
C1-T2       s2     False    memory_llm   .RRR.RRR.. 33 33 33 33 33 0.667 0.600 1.000 1.000   0.500
C1-T2     

### (f) Chunk-level Recall@K (extra)

This is the textbook definition: relevant chunks found in the top K divided by all relevant chunks in that collection. These numbers are small because some papers have dozens of relevant chunks, so even a perfect top 5 can't reach 1.0. They also aren't comparable between strategies, because the two collections have different numbers of relevant chunks. That's why I use the fact-level Recall@K in the main tables.

In [35]:
chunk_cols = [f"chunkR@{k}" for k in K_VALUES]
print(std.groupby("condition")[chunk_cols].mean().to_string(float_format=lambda x: f"{x:.3f}"))

                                    chunkR@3  chunkR@5
condition                                             
S1 fixed words | no rerank             0.186     0.320
S1 fixed words | rerank                0.193     0.296
S2 recursive sentences | no rerank     0.228     0.344
S2 recursive sentences | rerank        0.187     0.303


In [36]:
# save everything the tables are based on
run_rows = []
for (qid, variant, s, rr), hits in RUNS.items():
    for h in hits:
        rel, facts = judge(qid, h)
        run_rows.append({"query": qid, "variant": variant, "query_text": QUERY_USED[(qid, variant)], "strategy": s,
                         "reranked": rr, "rank": h["rank"], "chroma_rank": h.get("chroma_rank", h["rank"]),
                         "chunk_id": h["chunk_id"], "doc_id": h["doc_id"], "distance": h["distance"],
                         "ce_score": h.get("ce_score"), "relevant": rel, "facts": " | ".join(facts),
                         "text_start": h["text"][:200]})
suffix = "_smoke" if SMOKE_TEST else ""
pd.DataFrame(run_rows).to_csv(f"results/retrieval_runs{suffix}.csv", index=False)
results.to_csv(f"results/metrics_per_run{suffix}.csv", index=False)

metrics_json = {
    "config": CONFIG, "embed_model_used": EMBED_USED, "rerank_model_used": RERANK_USED, "llm_available": LLM_OK,
    "device": f"{device} ({device_name})",
    "chunk_stats": json.loads(chunk_stats.to_json(orient="index")),
    "token_stats": token_rows, "index_time": INDEX_TIME,
    "table_a_conditions": json.loads(table_a.to_json(orient="index")),
    "table_e_memory_mean": json.loads(table_e_mean.to_json(orient="index")),
    "rewrites": REWRITES, "answers": {f"{k[0]}|{k[1]}": v for k, v in ANSWERS.items()},
    "queries_used": {f"{k[0]}|{k[1]}": v for k, v in QUERY_USED.items()},
    "total_runtime_s": round(time.time() - RUN_START, 1),
}
with open(f"results/hw10_metrics{suffix}.json", "w") as f:
    json.dump(metrics_json, f, indent=1, default=str)
log(f"saved results/retrieval_runs{suffix}.csv, results/metrics_per_run{suffix}.csv, results/hw10_metrics{suffix}.json")
log(f"total notebook time so far: {(time.time() - RUN_START) / 60:.1f} min")

[2026-11-10T19:36:48Z] saved results/retrieval_runs.csv, results/metrics_per_run.csv, results/hw10_metrics.json
[2026-11-10T19:36:48Z] total notebook time so far: 37.4 min


## 17. Comparison and Discussion

All numbers below come from the tables in Section 16. K = 3 and K = 5, MRR is MRR@10. The standalone results are averaged over 8 queries, and the memory results over 2 follow-ups × 2 strategies × 2 rerank settings = 8 ranked lists per memory variant.

### Mean results (standalone queries)

| Condition | P@3 | P@5 | R@3 | R@5 | MRR@10 |
| --- | ---: | ---: | ---: | ---: | ---: |
| S1 fixed words, no rerank | 0.833 | 0.850 | 0.885 | 0.958 | 0.938 |
| S1 fixed words, rerank | 0.875 | 0.825 | 0.917 | 0.958 | 0.917 |
| S2 recursive sentences, no rerank | **0.917** | **0.850** | 0.844 | 0.958 | **1.000** |
| S2 recursive sentences, rerank | 0.833 | 0.800 | 0.885 | 0.958 | 0.938 |

### First, the retrieval was already easy at the paper level

The `top5 papers` column in Section 15 shows that **in all 32 standalone ranked lists, all top 5 chunks came from the correct paper**. The 100 papers cover very different topics (gravitational waves, turbo decoders, phylogenetics, ...), and my questions use words from their papers, so mxbai has no trouble finding the right paper. So the real question for every method is *which chunks of the right paper* end up on top: a chunk that actually states the fact, or one with the reference list, a proof step or the introduction. That's why many of the numbers are close to 1.0, and also why the differences between the methods are small.

### Chunking strategy 1 vs. strategy 2

- **Without reranking, S2 (sentence-aware) was slightly better:** P@3 0.917 vs. 0.833 and MRR 1.000 vs. 0.938. P@5 and R@5 were tied. The MRR difference comes from **one query (Q6)**: S1's top chunk for the magnetic-tunnel-junction question was a chunk from the right paper with none of the facts, and the first relevant chunk was at rank 2. S1 had a slightly higher R@3 (0.885 vs. 0.844).
- **With reranking it flips a little:** S1 is slightly better on P@3/P@5/R@3, and S2 still has the higher MRR (0.938 vs. 0.917).
- Per query, MRR@10 was the same for both strategies in 14 of 16 (query, rerank) pairs, and S2 was better in the other 2.

**My take:** sentence-aware chunking worked a little better, mainly at the very top of the ranking (P@3 and MRR without reranking). That makes sense, because 97.6% of S2's chunks end at a sentence boundary compared to 7.0% for S1. An S2 chunk is a set of complete sentences, which is closer to what mxbai was trained on, and the important sentence isn't cut in half. S2 also has fewer chunks over the 512-token limit (3.9% vs. 5.3%), so less text is ignored when embedding. But the gap is one or two queries out of eight, so I wouldn't call it a clear win. S2 also has downsides I could see in the data: 24 tiny leftover chunks under 30 words (one is just `"shapiro,"`), and 21% of neighbouring S2 chunks have no overlap at all.

### Did reranking help?

**Not overall.** It lowered P@5 for both strategies (S1 0.850 → 0.825, S2 0.850 → 0.800) and MRR@10 (S1 0.938 → 0.917, S2 1.000 → 0.938). It changed nothing for R@5 (0.958 everywhere). Per (query, strategy) pair, reranking improved P@5 in 3 cases, left it the same in 9 and made it worse in 4. For MRR it was better in 1, the same in 13 and worse in 2.

The one thing it did improve is **R@3**, for both strategies (S1 0.885 → 0.917, S2 0.844 → 0.885). The clearest example is Q1, where R@5 went from 0.667 to 1.000 for both strategies: the cross-encoder moved a chunk that covers the third fact (the SVD basis removes the bias of the phenomenological templates and matches the NR waveforms better) into the top 5. ChromaDB had ranked that chunk lower. So it put a *wider set of facts* near the top, but it also brought in more chunks that don't answer the question.

Where it hurt:
- **Q5 (galled networks):** MRR dropped from 1.000 to 0.333 (S1) and 0.500 (S2). The reranker's new #1 for S1 is a proof step ("at most two nodes in [math] are mapped to the same node ...") that ChromaDB had at rank 10. It has the words "at most" and "nodes", but it doesn't answer the question. In S2, its #1 was a background chunk about reticulation visible networks in general.
- **Q2 (turbo decoder):** P@5 dropped from 1.0 to 0.6 (S1) because the reranker moved the paper's general introduction and a reference-list chunk into the top 5.

I think there are two reasons. (1) The ceiling effect above: the bi-encoder already got the right paper every time, so there wasn't much left to fix. (2) The reranker (`ms-marco-MiniLM-L6-v2`) was trained on web search passages, not on math-heavy physics text full of `[math]` placeholders, and it seems to react to matching words like "at most" or "throughput" more than to whether the chunk actually answers the question. A bigger or scientific-domain reranker might do better, but I didn't test that.

### Did conversational memory help with the follow-up query?

| Follow-up variant | P@3 | P@5 | R@3 | R@5 | MRR@10 |
| --- | ---: | ---: | ---: | ---: | ---: |
| no memory | 0.667 | 0.500 | 0.625 | 0.708 | 0.854 |
| memory (concat previous question) | **0.833** | **0.800** | **0.750** | **0.917** | **0.938** |
| memory (LLM rewrite) | 0.667 | 0.550 | 0.708 | **0.917** | 0.875 |

**Yes, but less than I expected, and the simple version was the better one.** Both memory variants raised R@5 from 0.708 to 0.917. The plain concatenation was the best on every metric: P@5 went from 0.500 to 0.800.

Why the gain isn't bigger: my follow-ups still contained some topic words ("cavity", "laser light", "prior matrix"). In such a small collection those words alone often found the right paper. For C1-T2, even without memory, all top 5 chunks came from paper 33. The no-memory query really failed in one case: C2-T2 ("Does this approach need a prior matrix to start from?") on S2 without reranking. There, a chunk from an unrelated paper got into the top 5 (paper 93, a radio-telescope imaging paper, with a chunk about an ill-conditioned matrix), and only 1 of the top 5 chunks was relevant (P@5 0.2, MRR 0.333). With the concat memory the same setting got P@5 = 1.0 and MRR = 1.0.

How the queries were interpreted is shown in Section 13:
- C1-T2 was rewritten to *"Why does injecting a squeezed vacuum field into a detuned cavity work better than probing the cavity with ordinary laser light?"* That's exactly the intended meaning.
- C2-T2 was rewritten to *"Does the gravity-type model require a prior origin-destination matrix ...?"* That's **partly wrong**: the gravity model is something from the paper's literature review that the LLM had mentioned in its own turn-1 answer, not the paper's actual method. So the memory carried a mistake from the earlier answer into the search query. For C2-T2 on S1 without reranking this rewrite got P@3 = 0.333, worse than no memory (0.667). The concat version doesn't have this problem, because it only uses what the *user* typed.

For both follow-ups I judged relevance by what the user meant (Section 14), and the same labels were used for all three query versions, so the comparison is fair.

### What Precision@K, Recall@K and MRR show

- **MRR@10** was close to 1 for almost everything, so a relevant chunk was nearly always at rank 1. It only catches the few bad cases (Q5 after reranking, Q6 for S1, C2-T2 without memory). By itself it would make all methods look the same.
- **Precision@K** is the metric that separated the methods most. It shows how many of the chunks we'd actually give the LLM are useful. P@5 is lower than P@3 in most conditions, which means the useful chunks are near the top and the ranks 4-5 are more often off-topic chunks from the same paper.
- **Recall@K (facts)** is high (0.958 at K = 5): five chunks are usually enough to cover all the facts I listed. R@3 shows more differences, and that's where reranking helped.
- Taken together: for these queries the system finds the right paper and the most important fact easily. The weak point is that a good part of the top 5 is filler from the same paper, and neither chunking nor reranking fixed that reliably.

## 18. Limitations and Conclusion

### Limitations

- **Small sample.** Only the first 100 papers, mostly physics. Only 8 standalone queries and 2 follow-ups, so one query can change an average by 0.06-0.13. None of the differences above have been tested for significance, so they're trends at most.
- **I wrote the queries after reading the papers.** They use words from the papers, which makes dense retrieval look very good (the right paper was always in the top 5). Real users who don't know the paper's wording would make it harder.
- **Manual relevance rules.** One person (me) defined the target paper and the facts, and a chunk counts as relevant if it contains a key phrase. This is consistent and anyone can check it, but it's strict. For example, the S2 Q5 chunk saying "reticulation visible networks include galled trees and galled networks" is counted as not relevant, and so is the C1-T2 chunk that only says the comparison with classical light comes in a later section. A different person could draw these lines differently.
- **Recall is fact-level.** I count facts and not chunks (Section 15), so these Recall@K numbers aren't directly comparable to papers that use chunk-level recall. The chunk-level version is in table (f).
- **Chunk size effects.** 5.3% of S1 chunks and 3.9% of S2 chunks are longer than mxbai's 512 tokens (the `[math]` placeholders cost about 3 tokens each, and one S1 chunk from the logic paper has 1,717 tokens), so their endings are ignored in the embedding. The reranker also cuts query + chunk at 512 tokens. S2 has 24 tiny fragments. I only tried one size per strategy (about 200 words), so I can't say how chunk size alone changes the results.
- **Models.** One embedding model, one small web-trained reranker, and one 3B LLM. The LLM rewrite shows that a small model can carry a mistake from its own earlier answer into the next query. The answers themselves were not scored.
- **The text is already processed** (lowercase, equations removed), so all three models see text that looks quite different from their training data.

### Conclusion

I built a local RAG pipeline on the first 100 papers of `ccdv/arxiv-summarization` (document config) based on the Local demo. It uses ChromaDB with two separate collections, `mxbai-embed-large-v1` embeddings, a `ms-marco-MiniLM-L6-v2` cross-encoder and `llama3.2:3b` through Ollama for memory. On these questions, **sentence-aware recursive chunking was slightly better than fixed word windows** (P@3 0.917 vs. 0.833, MRR 1.000 vs. 0.938 without reranking). **Reranking did not improve the results overall**: it got more different facts into the top 3 but lowered P@5 and MRR, mostly because it pushed proof steps and introduction chunks up for Q2 and Q5. **Conversational memory helped the follow-up questions**, and the simplest version (adding the previous user question to the query) worked best: P@5 0.500 → 0.800, R@5 0.708 → 0.917. The LLM rewrite resolved "that method" correctly in conversation 1 but brought a mistake from the previous answer into conversation 2. If I wanted to improve this further, I would try a larger or science-specific reranker, harder queries written without reading the papers, and different chunk sizes for both strategies.

### Cleaning up

The demo ends with `vector_store.delete_collection()` to save resources. I keep the two collections in `chroma_db/`, so the notebook can be re-run without embedding everything again (about 35 minutes on my M1). Setting `DELETE_COLLECTIONS = True` removes them.

In [37]:
DELETE_COLLECTIONS = False
if DELETE_COLLECTIONS:
    for store in STORES.values():
        store.delete_collection()
    log("deleted both collections")
else:
    print("collections kept in", CHROMA_DIR, {s: st._collection.count() for s, st in STORES.items()})

collections kept in chroma_db {'s1': 3411, 's2': 3500}
